# Pythia-410M vs Mamba-370M: next-token predictions side by side

This is the first step toward the embedding-graft experiment in `init.md`. It loads both base models, builds two
grafted versions of Pythia, and compares any of the four on the same input tokens: next-token top-k and sampled
continuations.
- `graft`: Pythia with Mamba's input embedding swapped in, no adapter.
- `graft+lsq`: `graft` plus an affine layer after the embedding, fit by least squares to map Mamba's embedding
  onto Pythia's.

**Why this pair**

| | `EleutherAI/pythia-410m` | `state-spaces/mamba-370m-hf` |
|---|---|---|
| family | Transformer (GPT-NeoX) | SSM (Mamba) |
| layers | 24 | 48 |
| d_model | 1024 | 1024 |
| pretraining data | the Pile, 300B tokens | the Pile, 300B tokens |
| post-training | none (base) | none (base) |
| embedding rows | 50304 | 50280 |
| `W_U` tied to `W_E`? | no (`embed_in` / `lm_head`, stored as `embed_out`) | yes (`lm_head` = `backbone.embeddings`) |

- **They share a tokenizer.** The two `tokenizer.json` files are identical: GPT-NeoX-20B BPE, the same
  50,254-entry vocab, 50,009 merges and 25 added tokens, and the same pre/post-processing. This was checked
  offline during setup, so the notebook loads one shared tokenizer.
- **The real vocab is ids 0..50276 (`V = 50277`).** Embedding rows past that are padding, so logits are sliced
  to `[:V]` and both distributions cover the same tokens.
- `<|endoftext|>` (id 0) is BOS, EOS and pad for both. Encoding adds no BOS or other special tokens.

**Setup.** Run this once. The venv is shared by all projects in `ai_safety/`.
```bash
cd ~/dev/research/ai_safety
uv venv --python 3.12 .venv
uv pip install --python .venv/bin/python torch transformers ipykernel ipywidgets jupyterlab
.venv/bin/python -m ipykernel install --user --name ai_safety --display-name "Python (ai_safety .venv)"
```
Then select the kernel **"Python (ai_safety .venv)"**. The first load downloads about 2.4 GB of weights to
`~/.cache/huggingface`.

**Where to run.** A laptop CPU is enough: the four models in fp32 take about 6.5 GB of RAM. The notebook also
runs unchanged on a GPU, for example `l40-worker`, because `DEVICE` picks `cuda` when it's available.

**Expected Mamba warnings on CPU.** Mamba's first forward pass prints that `causal_conv1d_fn` and
`selective_scan_fn` are "falling back to its reference PyTorch implementation". Its first `generate()` prints the
same for `causal_conv1d_update` and `selective_state_update`. This happens because the CUDA-only packages
`causal_conv1d` and `mamba_ssm` aren't installed. The outputs are correct; it's just slower. On a GPU box,
installing those two packages gives the fast kernels.

## Config

In [ ]:
import math
import torch

MODELS = {
    "pythia": ("EleutherAI/pythia-410m", "step143000"),   # final pretraining checkpoint
    "mamba":  ("state-spaces/mamba-370m-hf", "main"),
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32   # the models are small; fp32 also keeps Mamba's scan exact on CPU
EOS_ID = 0              # <|endoftext|>: BOS, EOS and pad for both models
V = 50277               # real vocab size; embedding rows past this are padding

## Environment

In [ ]:
import sys
import transformers, tokenizers

print("python       ", sys.version.split()[0], sys.executable)
print("torch        ", torch.__version__)
print("transformers ", transformers.__version__)
print("tokenizers   ", tokenizers.__version__)
print("device       ", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## Load both models

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# One shared tokenizer: the two models' tokenizer.json files are identical (see intro).
repo, rev = MODELS["pythia"]
tok = AutoTokenizer.from_pretrained(repo, revision=rev)

models = {
    name: AutoModelForCausalLM.from_pretrained(repo, revision=rev, dtype=DTYPE).to(DEVICE).eval()
    for name, (repo, rev) in MODELS.items()
}
torch.set_grad_enabled(False)

print(f"{'model':8} {'class':22} {'params':>7} {'d_model':>7} {'layers':>6} {'emb rows':>8}  tied W_U = W_E")
for name, m in models.items():
    W_E, W_U = m.get_input_embeddings().weight, m.get_output_embeddings().weight
    n_params = sum(p.numel() for p in m.parameters())   # tied weights are counted once
    print(f"{name:8} {type(m).__name__:22} {n_params / 1e6:>6.0f}M {m.config.hidden_size:>7} "
          f"{m.config.num_hidden_layers:>6} {W_E.shape[0]:>8}  {W_E.data_ptr() == W_U.data_ptr()}")

## Graft: Mamba's input embedding into Pythia
`graft` is a copy of Pythia whose input embedding rows `[:V]` are replaced by Mamba's, times `EMB_SCALE`. Everything
else is still Pythia's: all 24 blocks, the final LayerNorm and the unembedding (`lm_head`, stored as
`embed_out` in the checkpoint). There's no adapter
yet, so in init.md terms the adapter is the identity. Rows from `V` up are padding that's never looked up, so they
keep Pythia's values. The original `pythia` is untouched.

The two embeddings differ in two ways:
- **Scale.** Mamba's rows are about 8.4× larger (mean row norm 6.68 vs 0.79). Setting `EMB_SCALE` to the printed
  `pythia/mamba` ratio (about 0.118) fixes the scale.
- **Basis.** The cosine similarity between a token's Pythia row and its Mamba row averages 0.0009, so the
  coordinates mean unrelated things. No scalar fixes that; that's the adapter's job. The next section fits one.

With the raw swap (`EMB_SCALE = 1.0`), graft's next-token distributions are flat and dominated by punctuation
and newlines. Its greedy continuations are mostly newlines, and its samples are word salad.

After changing `EMB_SCALE`, re-run this cell, then the next one. Together they rebuild `graft` and
`graft+lsq` from a fresh copy of Pythia.

In [ ]:
import copy

EMB_SCALE = 1.0   # 1.0 = raw swap; the printed pythia/mamba ratio (~0.118) matches Pythia's mean row norm

pythia, mamba = models["pythia"], models["mamba"]
E_p = pythia.get_input_embeddings().weight   # [50304, 1024] gpt_neox.embed_in
E_m = mamba.get_input_embeddings().weight    # [50280, 1024] backbone.embeddings (tied to Mamba's lm_head)
P_norm, M_norm = E_p[:V].norm(dim=1).mean().item(), E_m[:V].norm(dim=1).mean().item()
print(f"mean row norm: pythia {P_norm:.3f}  mamba {M_norm:.3f}  pythia/mamba {P_norm / M_norm:.3f}")

graft = copy.deepcopy(pythia)
E_g = graft.get_input_embeddings().weight
E_g[:V] = EMB_SCALE * E_m[:V]
models["graft"] = graft

assert torch.equal(E_g[:V], EMB_SCALE * E_m[:V])
assert torch.equal(graft.get_output_embeddings().weight, pythia.get_output_embeddings().weight)
assert E_g.data_ptr() != E_p.data_ptr()   # the original pythia is untouched

## Graft + least-squares affine
`graft+lsq` is `graft` with one extra layer. Its input embedding becomes
`Sequential(grafted Mamba embedding, Linear(1024 → 1024, with bias))`. That's the adapter slot from init.md, as a
real `nn.Linear`. Here it isn't trained; it's solved in closed form. Everything after it is still Pythia's.

**The fit.** Write `E_m` for the grafted (Mamba) rows and `E_p` for Pythia's own rows, both `[:V]`. `W` and `b`
minimize the total squared error `Σ_t ‖E_m[t] W + b − E_p[t]‖²` over all 50,277 real tokens, each weighted equally.
`torch.linalg.lstsq` solves it in float64 on `[E_m, 1]`, where the column of ones gives the bias. `EMB_SCALE`
doesn't change the fit, because `W` absorbs any scalar.

**Reading the fit-quality numbers.** `fit[t] = E_m[t] W + b` is what Pythia's layers receive for token `t`, instead
of the `E_p[t]` they were trained on.
- **R²:** Pythia's rows are spread out around their average row, and that spread is what tells tokens apart. R² is
  the fraction of that spread the fit reproduces: 1 means all of it, 0 means none.
- **cosine:** the cosine of the angle between `fit[t]` and `E_p[t]`, averaged over tokens. It ignores length:
  1 means the same direction, 0 means unrelated. For the raw graft (Mamba row vs Pythia row) it's 0.0009.
- **nearest row:** compare `fit[t]` by cosine against all 50,277 Pythia rows. It counts as correct if the most
  similar one is token `t`'s own row. Chance is 1 in 50,277. It's measured on 2,000 random tokens.
- **norm:** the average length of `fit[t]` vs `E_p[t]`. When least squares can only partly predict the target, it
  pulls the fitted values toward the average row, so fitted rows come out shorter.

**Results:** R² ≈ 0.23, cosine ≈ 0.48, nearest row ≈ 90%, and norm ≈ 0.38 vs 0.79. The fit mostly keeps *which
token this is*, but loses most of *where exactly* Pythia puts it.

**What it does to the model** (observed with the default prompts):
- `graft+lsq` gets partway back. After "Once upon a time" it puts 0.52 on "," (Pythia: 0.53).
- After "The capital of France is", its top tokens are words a sentence could continue with (" a", " the",
  " to"), not graft's punctuation. But " Paris" isn't in its top 20.
- Greedy decoding degenerates into "the the the" and newlines, and samples are English-looking words with no coherence.

Re-run this cell after re-running the graft cell.

In [ ]:
import torch.nn.functional as F

X = torch.cat([E_g[:V], torch.ones(V, 1, device=DEVICE)], dim=1).double().cpu()   # grafted rows + bias column
Y = E_p[:V].double().cpu()                                                         # Pythia's own rows: the target
sol = torch.linalg.lstsq(X, Y).solution                                            # [1025, 1024]
W, b = sol[:-1], sol[-1]

affine = torch.nn.Linear(1024, 1024).to(DEVICE, DTYPE)
affine.weight.copy_(W.T)   # nn.Linear computes x @ weight.T + bias
affine.bias.copy_(b)

lsq = copy.deepcopy(graft)
lsq.set_input_embeddings(torch.nn.Sequential(lsq.get_input_embeddings(), affine))
models["graft+lsq"] = lsq

# Fit quality (float64, CPU).
fit = X @ sol
r2 = 1 - (fit - Y).pow(2).sum() / (Y - Y.mean(0)).pow(2).sum()
cos = F.cosine_similarity(fit, Y, dim=1)
idx = torch.randperm(V, generator=torch.Generator().manual_seed(0))[:2000]
nearest = (F.normalize(fit[idx], dim=1).float() @ F.normalize(Y, dim=1).float().T).argmax(dim=1)
print(f"R^2          {r2:.3f}   fraction of the spread of Pythia's rows that the fit reproduces")
print(f"cosine       {cos.mean():.3f}   mean cos(fit[t], pythia[t]); the raw graft is ~0.001")
print(f"nearest row  {(nearest == idx).double().mean():.3f}   fit[t] is closest to token t's own Pythia row (2000 tokens)")
print(f"norm         {fit.norm(dim=1).mean():.3f} vs {Y.norm(dim=1).mean():.3f}   mean row length, fit vs pythia")

# The new layer computes the fit; everything after it is still Pythia's; graft is untouched.
ids = torch.tensor([[0, 1, 510, 5428, V - 1]], device=DEVICE)
assert torch.allclose(lsq.get_input_embeddings()(ids)[0], fit[ids[0].cpu()].to(DEVICE, DTYPE), atol=1e-4)
assert torch.equal(lsq.get_output_embeddings().weight, pythia.get_output_embeddings().weight)
assert isinstance(graft.get_input_embeddings(), torch.nn.Embedding)

## Next-token predictions, side by side
`side_by_side(prompt, names)` compares the models you name, one column each, in that order. For example,
`side_by_side(prompt, ["pythia", "graft+lsq"])`, or `list(models)` for all of them. `models` holds `pythia`,
`mamba`, `graft` and `graft+lsq`. A single name also works.

The prompt is encoded once, with nothing added, and the same token ids go to each model. Each model gets one
forward pass. The last-position logits are cast to float32, sliced to the real vocab `[:V]`, and log-softmaxed.
`prepend_eos=True` puts `<|endoftext|>` in front to mark the start of a document.

BPE attaches the space to the *next* word, so a trailing space in the prompt changes the input tokens. The
`input tokens` line shows exactly what the models see.

In [ ]:
def pick(names):
    """['pythia', 'graft+lsq'] (or one name) -> {name: model} from `models`, in the given order."""
    names = [names] if isinstance(names, str) else list(names)
    unknown = [n for n in names if n not in models]
    assert not unknown, f"unknown model(s) {unknown}; available: {list(models)}"
    return {n: models[n] for n in names}

def side_by_side(prompt, names, k=20, prepend_eos=False):
    chosen = pick(names)
    ids = tok.encode(prompt, add_special_tokens=False)
    if prepend_eos:
        ids = [EOS_ID] + ids
    print("input tokens:", [tok.decode([i]) for i in ids])
    input_ids = torch.tensor([ids], device=DEVICE)

    tops = {}
    for name, m in chosen.items():
        logprobs = m(input_ids).logits[0, -1, :V].float().log_softmax(-1)
        tops[name] = torch.topk(logprobs, k)

    col = f"{'token':16} {'prob':>6} {'logprob':>8}"
    print(f"\n{'rank':>4}  " + "   |   ".join(f"{name:^{len(col)}}" for name in chosen))
    print(f"{'':>4}  " + "   |   ".join(col for _ in chosen))
    for r in range(k):
        cells = []
        for name in chosen:
            lp, i = tops[name].values[r].item(), tops[name].indices[r].item()
            cells.append(f"{tok.decode([i])!r:16.16} {math.exp(lp):>6.3f} {lp:>8.3f}")
        print(f"{r:>4}  " + "   |   ".join(cells))

side_by_side("The capital of France is", list(models))

## Sampling, side by side
`sample_side_by_side(prompt, names)` takes model names the same way as `side_by_side`.

Each model continues the same prompt twice. The first is greedy. The second is one seeded sample from the full
distribution (temperature 1, `top_k=0`, `top_p=1`). Every `generate()` argument is passed explicitly, because
otherwise transformers 5 samples with `top_k=50`.
- Padding ids (`V` and up) are suppressed, so they can't be emitted.
- `torch.manual_seed(seed)` runs right before each model's sample, so each column is reproducible no matter what
  the other models do.
- Generation stops early if a model emits `<|endoftext|>`.
- Newlines in the output are shown as `↵`, followed by a real line break.

In [ ]:
import itertools, textwrap

def wrap(text, width):
    # Word-wrap each line to `width`, keeping the text's own newlines (marked with ↵).
    lines = text.replace("\n", "↵\n").split("\n")
    return [row for line in lines for row in (textwrap.wrap(line, width) or [""])]

def print_columns(texts, width):
    cols = {name: wrap(t, width) for name, t in texts.items()}
    print(" | ".join(f"{name:<{width}}" for name in cols))
    print("-+-".join("-" * width for _ in cols))
    for row in itertools.zip_longest(*cols.values(), fillvalue=""):
        print(" | ".join(f"{cell:<{width}}" for cell in row))

def sample_side_by_side(prompt, names, max_new_tokens=40, seed=0, width=30):
    input_ids = torch.tensor([tok.encode(prompt, add_special_tokens=False)], device=DEVICE)
    greedy, sampled = {}, {}
    for name, m in pick(names).items():
        common = dict(
            attention_mask=torch.ones_like(input_ids),
            max_new_tokens=max_new_tokens,
            eos_token_id=EOS_ID,                                   # stop at end-of-document
            pad_token_id=EOS_ID,
            suppress_tokens=list(range(V, m.config.vocab_size)),   # never emit padding ids
        )
        greedy[name] = tok.decode(m.generate(input_ids, do_sample=False, **common)[0])
        torch.manual_seed(seed)
        out = m.generate(input_ids, do_sample=True, temperature=1.0, top_k=0, top_p=1.0, **common)
        sampled[name] = tok.decode(out[0])

    print("greedy\n")
    print_columns(greedy, width)
    print(f"\nsample (seed {seed}, temperature 1, no top-k / top-p)\n")
    print_columns(sampled, width)

sample_side_by_side("The capital of France is", list(models))

## Scratch

In [ ]:
side_by_side("Once upon a time", ["pythia", "graft+lsq"])
sample_side_by_side("Once upon a time", ["pythia", "graft", "graft+lsq"])